# Tutorial 8: Activation Patching

**Prerequisites:** T02 (Hooks), T03 (Residual Stream), T06 (First Investigation)

**What you'll learn:**
- What activation patching is and why it's the gold standard for causal analysis
- The difference between denoising and noising experiments
- How to patch at every level: layers, heads, positions, and individual dimensions
- How to use IRTK's patching API

---

## The Problem with Correlation

When we observe that a head has high attention to a particular position, we learn that the head *correlates* with the behavior. But correlation isn't causation.

**Activation patching** provides **causal evidence**: we literally replace an activation from one run with the activation from a different run and measure the effect on output.

```
Clean run:      "The cat sat on the" → predicts "mat"
Corrupted run:  "The dog sat on the" → predicts "bone"

Patching: Run the clean input, but at layer 3, swap in the
corrupted run's activation. If the prediction changes to "bone",
then layer 3 was carrying the "cat vs dog" signal.
```

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=3, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Two inputs that differ at position 2
clean_tokens     = jnp.array([10, 20, 30, 40, 50])
corrupted_tokens = jnp.array([10, 20, 99, 40, 50])

logits_clean, cache_clean = model.run_with_cache(clean_tokens)
logits_corrupt, cache_corrupt = model.run_with_cache(corrupted_tokens)

# How different are the outputs?
diff = float(jnp.linalg.norm(logits_clean[-1] - logits_corrupt[-1]))
print(f'Clean tokens:     {list(np.array(clean_tokens))}')
print(f'Corrupted tokens: {list(np.array(corrupted_tokens))}')
print(f'Output difference at last position: {diff:.4f}')

## Denoising vs. Noising

There are two directions for patching:

1. **Denoising (corrupted → clean)**: Run the corrupted input, patch in clean activations
   - "Does restoring this activation fix the corruption?"
   - Measures: is this component **sufficient** to carry the signal?

2. **Noising (clean → corrupted)**: Run the clean input, patch in corrupted activations
   - "Does corrupting this activation break the clean behavior?"
   - Measures: is this component **necessary** for the signal?

In [ ]:
# Denoising: Run corrupted, patch in clean activations
# If it helps, this component CARRIES the signal

print('DENOISING: Patch clean activations into corrupted run')
print('(Higher recovery = component carries the clean signal)\n')

baseline_diff = diff  # Already computed above

for layer in range(cfg.n_layers):
    for component in ['hook_resid_pre', 'hook_attn_out', 'hook_mlp_out']:
        hook_name = f'blocks.{layer}.{component}'
        clean_act = cache_clean[hook_name]
        
        # Patch clean activation into corrupted run
        hooks = {hook_name: lambda x, n, act=clean_act: act}
        logits_patched = model.run_with_hooks(corrupted_tokens, fwd_hooks=hooks)
        
        # How close to clean output?
        dist_to_clean = float(jnp.linalg.norm(logits_patched[-1] - logits_clean[-1]))
        recovery = (baseline_diff - dist_to_clean) / baseline_diff * 100
        
        name = component.replace('hook_', '')
        bar = '#' * max(0, int(recovery / 5))
        print(f'  L{layer} {name:10s}: {recovery:+6.1f}% {bar}')

In [ ]:
# Noising: Run clean, patch in corrupted activations
# If it hurts, this component is NEEDED for the clean signal

print('NOISING: Patch corrupted activations into clean run')
print('(More negative = component is more necessary)\n')

for layer in range(cfg.n_layers):
    for component in ['hook_attn_out', 'hook_mlp_out']:
        hook_name = f'blocks.{layer}.{component}'
        corrupt_act = cache_corrupt[hook_name]
        
        # Patch corrupted activation into clean run
        hooks = {hook_name: lambda x, n, act=corrupt_act: act}
        logits_patched = model.run_with_hooks(clean_tokens, fwd_hooks=hooks)
        
        # How far from clean output?
        dist_from_clean = float(jnp.linalg.norm(logits_patched[-1] - logits_clean[-1]))
        damage = dist_from_clean / baseline_diff * 100
        
        name = component.replace('hook_', '')
        bar = '!' * min(int(damage / 5), 20)
        print(f'  L{layer} {name:10s}: {damage:6.1f}% damage {bar}')

## Per-Head Patching

We can go finer-grained and patch individual attention heads, not just whole layers.

In [ ]:
# Per-head activation patching (denoising)
print('Per-head denoising: patch clean head output into corrupted run\n')

for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        # Patch this specific head's z vector
        clean_z = cache_clean[f'blocks.{layer}.attn.hook_z']
        
        def patch_head(z, name, h=head, clean=clean_z):
            return z.at[:, h, :].set(clean[:, h, :])
        
        hooks = {f'blocks.{layer}.attn.hook_z': patch_head}
        logits_patched = model.run_with_hooks(corrupted_tokens, fwd_hooks=hooks)
        
        dist = float(jnp.linalg.norm(logits_patched[-1] - logits_clean[-1]))
        recovery = (baseline_diff - dist) / baseline_diff * 100
        bar = '#' * max(0, int(recovery / 3))
        print(f'  L{layer}H{head}: {recovery:+6.1f}% {bar}')

## Position-Specific Patching

We can also patch at specific **positions** — this tells us which positions carry the signal.

In [ ]:
# Position-specific patching at the residual stream level
print('Position-specific denoising at each layer:\n')

for layer in range(cfg.n_layers):
    print(f'Layer {layer}:')
    for pos in range(len(clean_tokens)):
        clean_resid = cache_clean[f'blocks.{layer}.hook_resid_pre']
        
        def patch_pos(x, name, p=pos, clean=clean_resid):
            return x.at[p, :].set(clean[p, :])
        
        hooks = {f'blocks.{layer}.hook_resid_pre': patch_pos}
        logits_patched = model.run_with_hooks(corrupted_tokens, fwd_hooks=hooks)
        
        dist = float(jnp.linalg.norm(logits_patched[-1] - logits_clean[-1]))
        recovery = (baseline_diff - dist) / baseline_diff * 100
        bar = '#' * max(0, int(recovery / 3))
        marker = ' <-- changed token' if pos == 2 else ''
        print(f'    pos {pos} (token {int(clean_tokens[pos]):2d}): '
              f'{recovery:+6.1f}% {bar}{marker}')
    print()

## Using IRTK's Patching API

IRTK provides high-level functions that handle the boilerplate.

In [ ]:
from irtk.patching import activation_patch, zero_ablation, per_head_patch

# Define a metric: we'll track the logit of the clean prediction
clean_pred = int(jnp.argmax(logits_clean[-1]))
def metric(logits, token=clean_pred):
    return float(logits[-1, token])

# Patch at all attention and MLP outputs
hook_names = []
for l in range(cfg.n_layers):
    hook_names.extend([
        f'blocks.{l}.hook_attn_out',
        f'blocks.{l}.hook_mlp_out',
    ])

results = activation_patch(
    model, clean_tokens, corrupted_tokens,
    hook_names=hook_names,
    metric_fn=metric,
)

print(f'Metric: logit of token {clean_pred}')
print(f'Clean baseline: {metric(logits_clean):.4f}')
print(f'Corrupted baseline: {metric(logits_corrupt):.4f}')
print(f'\nPatching results (corrupted → clean):')
for name, value in sorted(results.items()):
    short = name.replace('blocks.', 'L').replace('.hook_', ' ')
    print(f'  {short:20s}: metric = {value:.4f}')

In [ ]:
# Per-head patching with the high-level API
head_results = per_head_patch(
    model, clean_tokens, corrupted_tokens,
    metric_fn=metric,
)

print('Per-head patching results:')
print(f'  Shape: {head_results.shape}  [n_layers, n_heads]')
print()
for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        val = float(head_results[layer, head])
        bar = '#' * max(0, int(abs(val) * 10))
        print(f'  L{layer}H{head}: {val:+.4f} {bar}')

## Zero Ablation vs. Mean Ablation

Besides patching between two runs, common ablation types include:

- **Zero ablation**: Replace with zeros ("what if this component didn't exist?")
- **Mean ablation**: Replace with the mean activation across a dataset ("what if this component gave its average output?")

Mean ablation is often preferred because it preserves the expected scale of the residual stream.

In [ ]:
# Zero ablation at each layer
results = zero_ablation(
    model, clean_tokens,
    hook_names=hook_names,
    metric_fn=metric,
)

clean_metric = metric(logits_clean)
print(f'Zero ablation (metric = logit of token {clean_pred}):')
print(f'Baseline: {clean_metric:.4f}\n')

for name, value in sorted(results.items()):
    change = value - clean_metric
    short = name.replace('blocks.', 'L').replace('.hook_', ' ')
    bar = '-' * min(int(abs(change) * 5), 20) if change < 0 else '+' * min(int(change * 5), 20)
    print(f'  {short:20s}: {value:.4f} (change: {change:+.4f}) {bar}')

## Key Takeaways

1. **Activation patching** is the gold standard for causal analysis in mechinterp
2. **Denoising** (corrupted→clean) tests if a component is **sufficient** to carry a signal
3. **Noising** (clean→corrupted) tests if a component is **necessary** for a behavior
4. **Granularity**: you can patch at the level of layers, heads, positions, or dimensions
5. **Zero and mean ablation** measure component importance without a corrupted input
6. **IRTK's patching API** handles the boilerplate: `activation_patch()`, `per_head_patch()`, `zero_ablation()`

**Next: [T09 — Circuit Discovery](T09_circuit_discovery.ipynb)** — Systematically finding the minimal circuit responsible for a behavior.